# MoMo-FDVS logical PR12 — clean-session preflight

This notebook mounts the project owner's Drive, checks out one immutable commit, installs repository lock files, inventories the runtime and validates generic paths. It does not acquire data or fit a model.

In [ ]:
RUN_PROFILE = "smoke"
TARGET_COMMIT = "cc3c59df047f10905392217d892f452bbd456771"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/00_environment_preflight.ipynb"
assert RUN_PROFILE == "smoke"
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR12_SHA"

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)

In [ ]:
import json
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report
from momo_fdvs_ml.execution import ExecutionProfile

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
report = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
print(json.dumps(report, indent=2, sort_keys=True))

## Stop boundary

A passing report proves only the clean Colab/runtime foundation. Do not acquire datasets or start reportable training. Continue to the tiny smoke notebook only after reviewing the recorded commit, paths and lock hashes.